# AI Research Agent — LangGraph map-reduce

**Flow:** `planner` → fan out via `Send()` → parallel `researcher` branches → `synthesizer`

Runs on a self-hosted `gpt-oss:20b` via Ollama. Requires `TAVILY_API_KEY` and `OLLAMA_BASE_URL` in `.env`.

> If you just installed/updated packages, **restart the kernel** before running these cells.

In [ ]:
# One-time setup (uncomment if needed), then RESTART THE KERNEL
# %pip install -U langchain langgraph langchain-ollama langchain-tavily python-dotenv

## Imports and LLM setup

In [ ]:
import operator
import os
from typing import Annotated

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv()

# base_url comes from .env so the notebook never hardcodes your LAN IP
llm = init_chat_model(
    model="ollama:gpt-oss:20b",
    temperature=0.2,  # low temp = reliable planning/synthesis
    base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434"),
)

search_tool = TavilySearch(max_results=3)

# quick connectivity check
llm.invoke("Reply with exactly: OK").content

## State

- `ResearchState` — the main graph state. `findings` uses a **reducer** (`operator.add`) so parallel branches append instead of overwriting each other.
- `ResearcherState` — the private input each `Send()` branch receives.

In [ ]:
class ResearchState(TypedDict):
    topic: str
    sub_questions: list[str]
    findings: Annotated[list[str], operator.add]  # reducer merges parallel branches
    report: str


class ResearcherState(TypedDict):
    sub_question: str

## Planner

Uses `.with_structured_output(..., method="json_schema")` — Ollama's native JSON schema mode, which avoids the `bind_tools` NotImplementedError path.

In [ ]:
class ResearchPlan(BaseModel):
    sub_questions: list[str] = Field(
        description="3-5 focused, independently-searchable sub-questions "
        "that together fully cover the research topic."
    )


def planner(state: ResearchState):
    """Break the topic into sub-questions using structured output."""
    planner_llm = llm.with_structured_output(ResearchPlan, method="json_schema")
    plan = planner_llm.invoke(
        f"You are a research planner. Break this topic into 3-5 focused "
        f"sub-questions that can each be answered with a web search:\n\n"
        f"Topic: {state['topic']}"
    )
    return {"sub_questions": plan.sub_questions}

## Fan-out + Researcher

`fan_out` returns a list of `Send()` objects from a conditional edge — LangGraph spawns one `researcher` branch per sub-question, in parallel.

In [ ]:
def fan_out(state: ResearchState):
    """Send() one researcher per sub-question — they run in parallel."""
    return [Send("researcher", {"sub_question": q}) for q in state["sub_questions"]]


def researcher(state: ResearcherState):
    """Search the web for one sub-question and summarize the results."""
    question = state["sub_question"]
    results = search_tool.invoke({"query": question})

    summary = llm.invoke(
        f"Summarize these search results into 2-3 dense paragraphs that "
        f"answer the question. Cite source URLs inline.\n\n"
        f"Question: {question}\n\nSearch results:\n{results}"
    )
    return {"findings": [f"### {question}\n\n{summary.content}"]}

## Synthesizer

In [ ]:
def synthesizer(state: ResearchState):
    """Merge all findings into one final markdown report."""
    findings_block = "\n\n---\n\n".join(state["findings"])
    report = llm.invoke(
        f"You are a research writer. Using ONLY the findings below, write a "
        f"well-organized markdown report on: {state['topic']}\n\n"
        f"Include an intro, sections per theme, and a conclusion. Keep the "
        f"inline source URLs.\n\nFindings:\n{findings_block}"
    )
    return {"report": report.content}

## Build and visualize the graph

In [ ]:
builder = StateGraph(ResearchState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("synthesizer", synthesizer)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", fan_out, ["researcher"])
builder.add_edge("researcher", "synthesizer")
builder.add_edge("synthesizer", END)

graph = builder.compile()

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## Run it

In [ ]:
topic = "impact of solid state batteries on EVs"  # <-- change me

result = graph.invoke({"topic": topic, "findings": []})

print("Sub-questions researched:")
for q in result["sub_questions"]:
    print(f"  - {q}")

In [ ]:
from IPython.display import Markdown
Markdown(result["report"])

In [ ]:
# Save the report to disk
with open("report.md", "w") as f:
    f.write(result["report"])
print("Report saved to report.md")